In [ ]:
# ============================================================
# STEP 0 — PROJECT ENVIRONMENT CHECK
# ============================================================

import os
import sys
import numpy as np
import cv2
import sklearn
import skimage

print("===== ENVIRONMENT =====")
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Scikit-image:", skimage.__version__)

print("\n===== /content FILES =====")

for item in sorted(os.listdir("/content")):
    print(item)

print("\n===== PROJECT FOLDERS =====")

for item in sorted(os.listdir("/content")):
    path = os.path.join("/content", item)
    if os.path.isdir(path):
        print("[FOLDER]", item)

print("\n===== CHECK COMPLETE =====")
print("No model training was performed.")

In [ ]:
# ============================================================
# STEP 1 — FIX AND VERIFY BSDS500 PATHS
# ============================================================

import os

DATA_DIR =  "/content/BSR/BSDS500"

TRAIN_DIR = os.path.join(DATA_DIR, "data", "images", "train")
VAL_DIR   = os.path.join(DATA_DIR, "data", "images", "val")
TEST_DIR  = os.path.join(DATA_DIR, "data", "images", "test")

def get_images(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

train_images = get_images(TRAIN_DIR)
val_images   = get_images(VAL_DIR)
test_images  = get_images(TEST_DIR)

print("===== BSDS500 PATHS =====")
print("Train      :", TRAIN_DIR)
print("Validation :", VAL_DIR)
print("Test       :", TEST_DIR)

print("\n===== IMAGE COUNTS =====")
print("Train      :", len(train_images))
print("Validation :", len(val_images))
print("Test       :", len(test_images))
print("Total      :", len(train_images) + len(val_images) + len(test_images))

print("\n===== FIRST 3 TRAIN IMAGES =====")
for path in train_images[:3]:
    print(path)

print("\n===== FIRST 3 VALIDATION IMAGES =====")
for path in val_images[:3]:
    print(path)

print("\n===== FIRST 3 TEST IMAGES =====")
for path in test_images[:3]:
    print(path)

print("\n===== STEP 1 VERIFIED =====")

In [ ]:
# ============================================================
# STEP 2 — INSPECT BSDS500 IMAGES
# ============================================================

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Load one image from each split
train_img = cv2.imread(train_images[0])
val_img   = cv2.imread(val_images[0])
test_img  = cv2.imread(test_images[0])

# OpenCV loads BGR, so convert to RGB
train_img = cv2.cvtColor(train_img, cv2.COLOR_BGR2RGB)
val_img   = cv2.cvtColor(val_img, cv2.COLOR_BGR2RGB)
test_img  = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

print("===== IMAGE INFORMATION =====")

print("TRAIN")
print("File :", train_images[0])
print("Shape:", train_img.shape)
print("Dtype:", train_img.dtype)

print("\nVALIDATION")
print("File :", val_images[0])
print("Shape:", val_img.shape)
print("Dtype:", val_img.dtype)

print("\nTEST")
print("File :", test_images[0])
print("Shape:", test_img.shape)
print("Dtype:", test_img.dtype)

# ------------------------------------------------------------
# Display images
# ------------------------------------------------------------

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(train_img)
plt.title("Training Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(val_img)
plt.title("Validation Image")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(test_img)
plt.title("Test Image")
plt.axis("off")

plt.tight_layout()
plt.show()

print("\n===== STEP 2 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 3 — RGB → GRAYSCALE → YUV
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Use the first training image
image_path = train_images[0]

# ------------------------------------------------------------
# 1. Read original RGB image
# ------------------------------------------------------------

image_bgr = cv2.imread(image_path)

if image_bgr is None:
    raise ValueError("Could not load image.")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# ------------------------------------------------------------
# 2. Resize while keeping our project width = 400
# ------------------------------------------------------------

target_width = 400

original_h, original_w = image_rgb.shape[:2]

scale = target_width / original_w
target_height = int(original_h * scale)

image_rgb = cv2.resize(
    image_rgb,
    (target_width, target_height),
    interpolation=cv2.INTER_AREA
)

# ------------------------------------------------------------
# 3. Normalize RGB to 0–1
# ------------------------------------------------------------

image_float = image_rgb.astype(np.float32) / 255.0

# ------------------------------------------------------------
# 4. Convert RGB → YUV
# ------------------------------------------------------------

image_yuv = cv2.cvtColor(image_float, cv2.COLOR_RGB2YUV)

Y = image_yuv[:, :, 0]
U = image_yuv[:, :, 1]
V = image_yuv[:, :, 2]

# ------------------------------------------------------------
# 5. Create grayscale image from Y
# ------------------------------------------------------------

gray_from_Y = Y

# ------------------------------------------------------------
# 6. Print information
# ------------------------------------------------------------

print("===== PREPROCESSING INFORMATION =====")

print("Original shape :", image_rgb.shape)

print("\nY channel")
print("Shape :", Y.shape)
print("Min   :", Y.min())
print("Max   :", Y.max())
print("Mean  :", Y.mean())

print("\nU channel")
print("Shape :", U.shape)
print("Min   :", U.min())
print("Max   :", U.max())
print("Mean  :", U.mean())

print("\nV channel")
print("Shape :", V.shape)
print("Min   :", V.min())
print("Max   :", V.max())
print("Mean  :", V.mean())

# ------------------------------------------------------------
# 7. Display RGB and Y
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original Color Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gray_from_Y, cmap="gray")
plt.title("Y Channel (What Model Sees)")
plt.axis("off")

plt.tight_layout()
plt.show()

print("\n===== STEP 3 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 4 — SLIC SUPERPIXELS
# ============================================================

from skimage.segmentation import slic
import matplotlib.pyplot as plt
import numpy as np

# Use the RGB image from Step 3
# image_rgb already exists

N_SEGMENTS = 100

segments = slic(
    image_rgb,
    n_segments=N_SEGMENTS,
    compactness=10,
    start_label=0
)

number_of_segments = len(np.unique(segments))

print("===== SLIC INFORMATION =====")
print("Requested segments :", N_SEGMENTS)
print("Actual segments    :", number_of_segments)
print("Image shape        :", image_rgb.shape)
print("Segment map shape  :", segments.shape)

# ------------------------------------------------------------
# Display superpixel boundaries
# ------------------------------------------------------------

from skimage.segmentation import mark_boundaries

segmented_view = mark_boundaries(
    image_rgb,
    segments,
    color=(1, 0, 0)
)

plt.figure(figsize=(10, 6))
plt.imshow(segmented_view)
plt.title("SLIC Superpixels")
plt.axis("off")
plt.show()

print("\n===== STEP 4 COMPLETE =====")

In [ ]:
# ============================================
# STEP 5 — FEATURE EXTRACTION
# ============================================

from scipy.fft import fft2, fftshift
import numpy as np

PATCH_SIZE = 10

def extract_features_for_segment(Y, segments, segment_id):
    """
    Extract features from one SLIC superpixel.

    Features:
    1. 10x10 local luminance patch
    2. 2D FFT information
    3. Mean luminance
    4. Standard deviation
    5. Minimum luminance
    6. Maximum luminance
    7. Luminance histogram
    8. Gradient information
    9. Spatial position
    """

    # ----------------------------------------
    # Find pixels belonging to this segment
    # ----------------------------------------
    ys, xs = np.where(segments == segment_id)

    if len(xs) == 0:
        return None

    # ----------------------------------------
    # Find center of the segment
    # ----------------------------------------
    center_y = int(np.mean(ys))
    center_x = int(np.mean(xs))

    # ----------------------------------------
    # Extract local 10x10 patch
    # ----------------------------------------
    half = PATCH_SIZE // 2

    y1 = max(0, center_y - half)
    y2 = min(Y.shape[0], center_y + half)

    x1 = max(0, center_x - half)
    x2 = min(Y.shape[1], center_x + half)

    patch = Y[y1:y2, x1:x2]

    # ----------------------------------------
    # Resize patch to exactly 10x10
    # ----------------------------------------
    patch_10x10 = cv2.resize(
        patch,
        (PATCH_SIZE, PATCH_SIZE),
        interpolation=cv2.INTER_AREA
    )

    # ----------------------------------------
    # FFT features
    # ----------------------------------------
    fft_result = fft2(patch_10x10)
    fft_shifted = fftshift(fft_result)

    fft_magnitude = np.abs(fft_shifted)

    # Normalize FFT
    fft_magnitude = fft_magnitude / (np.max(fft_magnitude) + 1e-8)

    fft_features = fft_magnitude.flatten()

    # ----------------------------------------
    # Luminance statistics
    # ----------------------------------------
    mean_y = np.mean(patch)
    std_y = np.std(patch)
    min_y = np.min(patch)
    max_y = np.max(patch)

    statistics = np.array([
        mean_y,
        std_y,
        min_y,
        max_y
    ])

    # ----------------------------------------
    # Luminance histogram
    # ----------------------------------------
    histogram, _ = np.histogram(
        patch,
        bins=8,
        range=(0, 1),
        density=True
    )

    # Normalize histogram
    histogram = histogram / (np.sum(histogram) + 1e-8)

    # ----------------------------------------
    # Gradient / edge information
    # ----------------------------------------
    grad_x = cv2.Sobel(
        patch_10x10,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    grad_y = cv2.Sobel(
        patch_10x10,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    gradient_magnitude = np.sqrt(
        grad_x**2 + grad_y**2
    )

    gradient_features = np.array([
        np.mean(gradient_magnitude),
        np.std(gradient_magnitude)
    ])

    # ----------------------------------------
    # Spatial position
    # ----------------------------------------
    height, width = Y.shape

    position_features = np.array([
        center_x / width,
        center_y / height
    ])

    # ----------------------------------------
    # Combine everything
    # ----------------------------------------
    features = np.concatenate([
        patch_10x10.flatten(),      # 100
        fft_features,               # 100
        statistics,                 # 4
        histogram,                  # 8
        gradient_features,          # 2
        position_features           # 2
    ])

    return features


# ============================================
# Extract features from all segments
# ============================================

segment_ids = np.unique(segments)

all_features = []

for segment_id in segment_ids:

    features = extract_features_for_segment(
        Y,
        segments,
        segment_id
    )

    if features is not None:
        all_features.append(features)

X_demo = np.array(all_features)


# ============================================
# Information
# ============================================

print("===== FEATURE EXTRACTION INFORMATION =====")
print("Number of segments :", len(segment_ids))
print("Feature matrix shape:", X_demo.shape)
print("Features per segment:", X_demo.shape[1])

print("\nFeature breakdown:")
print("10x10 patch        : 100")
print("FFT features       : 100")
print("Statistics         : 4")
print("Histogram          : 8")
print("Gradient features  : 2")
print("Position features  : 2")
print("-------------------------------------------")
print("TOTAL              :", 216)

print("\n===== STEP 5 COMPLETE =====")

In [ ]:
# ============================================
# STEP 6 — CREATE U/V TARGETS
# ============================================

# Make sure U and V come from the original
# color image converted to YUV.
# Y, U and V were created in Step 3.

segment_ids = np.unique(segments)

target_U = []
target_V = []

for segment_id in segment_ids:

    # Find all pixels belonging to this superpixel
    mask = (segments == segment_id)

    # Average chrominance values inside the segment
    mean_u = np.mean(U[mask])
    mean_v = np.mean(V[mask])

    target_U.append(mean_u)
    target_V.append(mean_v)

# Convert to NumPy arrays
target_U = np.array(target_U)
target_V = np.array(target_V)

print("===== U/V TARGET INFORMATION =====")
print("Number of segments :", len(segment_ids))

print("\nU target:")
print("Shape :", target_U.shape)
print("Min   :", target_U.min())
print("Max   :", target_U.max())
print("Mean  :", target_U.mean())

print("\nV target:")
print("Shape :", target_V.shape)
print("Min   :", target_V.min())
print("Max   :", target_V.max())
print("Mean  :", target_V.mean())

print("\n===== CHECK =====")
print("Feature rows :", X_demo.shape[0])
print("U target rows:", target_U.shape[0])
print("V target rows:", target_V.shape[0])

if X_demo.shape[0] == target_U.shape[0] == target_V.shape[0]:
    print("\n✓ Every feature vector has one U target and one V target.")
else:
    print("\n✗ ERROR: Number of features and targets do not match.")

print("\n===== STEP 6 COMPLETE =====")

In [ ]:
# ============================================
# STEP 7 — IMPROVED FEATURE EXTRACTION
# ============================================

import numpy as np
import cv2
from skimage.segmentation import slic

IMAGE_WIDTH = 400
N_SEGMENTS = 100
PATCH_SIZE = 10

def extract_image_training_data(
    image_path,
    image_width=400,
    n_segments=100
):

    # ----------------------------------------
    # 1. Load image
    # ----------------------------------------

    image_bgr = cv2.imread(image_path)

    if image_bgr is None:
        raise ValueError(f"Could not load image: {image_path}")

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    # ----------------------------------------
    # 2. Resize
    # ----------------------------------------

    height, width = image_rgb.shape[:2]

    new_height = int(
        height * image_width / width
    )

    image_rgb = cv2.resize(
        image_rgb,
        (image_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    image_float = (
        image_rgb.astype(np.float32) / 255.0
    )

    # ----------------------------------------
    # 3. RGB -> YUV
    # ----------------------------------------

    image_yuv = cv2.cvtColor(
        image_float,
        cv2.COLOR_RGB2YUV
    )

    Y = image_yuv[:, :, 0]
    U = image_yuv[:, :, 1]
    V = image_yuv[:, :, 2]

    # ----------------------------------------
    # 4. SLIC using ONLY luminance
    # ----------------------------------------

    segments = slic(
        Y,
        n_segments=n_segments,
        compactness=10,
        start_label=0,
        channel_axis=None
    )

    segment_ids = np.unique(segments)

    # ----------------------------------------
    # 5. Improved image features
    # ----------------------------------------

    # Gradient
    grad_x = cv2.Sobel(
        Y,
        cv2.CV_32F,
        1,
        0,
        ksize=3
    )

    grad_y = cv2.Sobel(
        Y,
        cv2.CV_32F,
        0,
        1,
        ksize=3
    )

    gradient_magnitude = np.sqrt(
        grad_x ** 2 + grad_y ** 2
    )

    # Laplacian = local texture/detail
    laplacian = cv2.Laplacian(
        Y,
        cv2.CV_32F
    )

    # ----------------------------------------
    # 6. Build feature vectors
    # ----------------------------------------

    X = []
    target_U = []
    target_V = []

    for segment_id in segment_ids:

        mask = (
            segments == segment_id
        )

        ys, xs = np.where(mask)

        if len(xs) == 0:
            continue

        # ------------------------------------
        # Segment centroid
        # ------------------------------------

        center_y = int(np.mean(ys))
        center_x = int(np.mean(xs))

        # ------------------------------------
        # 10x10 local luminance patch
        # ------------------------------------

        half = PATCH_SIZE // 2

        y_start = max(
            0,
            center_y - half
        )

        y_end = min(
            Y.shape[0],
            center_y + half
        )

        x_start = max(
            0,
            center_x - half
        )

        x_end = min(
            Y.shape[1],
            center_x + half
        )

        patch = Y[
            y_start:y_end,
            x_start:x_end
        ]

        # Pad if near image boundary
        padded_patch = np.zeros(
            (PATCH_SIZE, PATCH_SIZE),
            dtype=np.float32
        )

        ph, pw = patch.shape

        padded_patch[
            :ph,
            :pw
        ] = patch

        # ------------------------------------
        # Normalize patch
        # ------------------------------------

        patch_mean = np.mean(
            padded_patch
        )

        patch_std = np.std(
            padded_patch
        ) + 1e-6

        normalized_patch = (
            padded_patch - patch_mean
        ) / patch_std

        # ------------------------------------
        # FFT features
        # ------------------------------------

        fft = np.fft.fft2(
            normalized_patch
        )

        fft_shifted = np.fft.fftshift(
            fft
        )

        fft_magnitude = np.abs(
            fft_shifted
        )

        fft_magnitude = fft_magnitude / (
            np.max(fft_magnitude) + 1e-6
        )

        # ------------------------------------
        # Multi-scale local patches
        # ------------------------------------

        local_5 = Y[
            max(0, center_y-2):
            min(Y.shape[0], center_y+3),
            max(0, center_x-2):
            min(Y.shape[1], center_x+3)
        ]

        local_15 = Y[
            max(0, center_y-7):
            min(Y.shape[0], center_y+8),
            max(0, center_x-7):
            min(Y.shape[1], center_x+8)
        ]

        # ------------------------------------
        # Local statistics
        # ------------------------------------

        segment_Y = Y[mask]

        mean_Y = np.mean(segment_Y)
        std_Y = np.std(segment_Y)
        min_Y = np.min(segment_Y)
        max_Y = np.max(segment_Y)

        # Percentiles
        p10 = np.percentile(
            segment_Y,
            10
        )

        p25 = np.percentile(
            segment_Y,
            25
        )

        p50 = np.percentile(
            segment_Y,
            50
        )

        p75 = np.percentile(
            segment_Y,
            75
        )

        p90 = np.percentile(
            segment_Y,
            90
        )

        # ------------------------------------
        # Histogram
        # ------------------------------------

        histogram, _ = np.histogram(
            segment_Y,
            bins=16,
            range=(0, 1),
            density=False
        )

        histogram = (
            histogram.astype(np.float32)
            / max(len(segment_Y), 1)
        )

        # ------------------------------------
        # Edge / texture statistics
        # ------------------------------------

        segment_gradient = (
            gradient_magnitude[mask]
        )

        segment_laplacian = (
            laplacian[mask]
        )

        gradient_mean = np.mean(
            segment_gradient
        )

        gradient_std = np.std(
            segment_gradient
        )

        gradient_max = np.max(
            segment_gradient
        )

        texture_mean = np.mean(
            np.abs(segment_laplacian)
        )

        texture_std = np.std(
            segment_laplacian
        )

        # ------------------------------------
        # Multi-scale statistics
        # ------------------------------------

        local_5_mean = np.mean(
            local_5
        )

        local_5_std = np.std(
            local_5
        )

        local_15_mean = np.mean(
            local_15
        )

        local_15_std = np.std(
            local_15
        )

        # ------------------------------------
        # Spatial position
        # ------------------------------------

        norm_x = (
            center_x / Y.shape[1]
        )

        norm_y = (
            center_y / Y.shape[0]
        )

        # ------------------------------------
        # Combine features
        # ------------------------------------

        features = np.concatenate([

            # Original paper-inspired features
            padded_patch.flatten(),
            fft_magnitude.flatten(),

            # Basic luminance
            np.array([
                mean_Y,
                std_Y,
                min_Y,
                max_Y
            ]),

            # Percentiles
            np.array([
                p10,
                p25,
                p50,
                p75,
                p90
            ]),

            # Histogram
            histogram,

            # Gradient
            np.array([
                gradient_mean,
                gradient_std,
                gradient_max
            ]),

            # Texture
            np.array([
                texture_mean,
                texture_std
            ]),

            # Multi-scale information
            np.array([
                local_5_mean,
                local_5_std,
                local_15_mean,
                local_15_std
            ]),

            # Position
            np.array([
                norm_x,
                norm_y
            ])
        ])

        X.append(features)

        # ------------------------------------
        # Target U/V
        # ------------------------------------

        target_U.append(
            np.mean(U[mask])
        )

        target_V.append(
            np.mean(V[mask])
        )

    return (
        np.array(X, dtype=np.float32),
        np.array(target_U, dtype=np.float32),
        np.array(target_V, dtype=np.float32),
        segments
    )


print("===== STEP 7 COMPLETE =====")

In [ ]:
# ============================================
# STEP 8 — CHECK IMPROVED FEATURES
# ============================================

X_check, U_check, V_check, segments_check = (
    extract_image_training_data(
        train_images[0],
        image_width=400,
        n_segments=100
    )
)

print("===== STEP 8 — FEATURE CHECK =====")

print("Number of segments :", X_check.shape[0])
print("Features per segment :", X_check.shape[1])
print("X shape :", X_check.shape)
print("U shape :", U_check.shape)
print("V shape :", V_check.shape)

print("\n===== STEP 8 COMPLETE =====")

In [ ]:
# ============================================
# STEP 9 — BUILD IMPROVED TRAINING DATASET
# ============================================

import time

print("===== STEP 9 — BUILDING TRAINING DATA =====")

start_time = time.time()

X_train = []
U_train = []
V_train = []

for i, image_path in enumerate(train_images):

    X_img, U_img, V_img, _ = extract_image_training_data(
        image_path,
        image_width=400,
        n_segments=100
    )

    X_train.append(X_img)
    U_train.append(U_img)
    V_train.append(V_img)

    if (i + 1) % 25 == 0:
        print(
            f"Processed {i + 1}/{len(train_images)} images"
        )

# Combine all images
X_train = np.vstack(X_train)
U_train = np.concatenate(U_train)
V_train = np.concatenate(V_train)

elapsed = time.time() - start_time

print("\n===== TRAINING DATA COMPLETE =====")

print("X_train shape :", X_train.shape)
print("U_train shape :", U_train.shape)
print("V_train shape :", V_train.shape)

print("\nTraining samples :", X_train.shape[0])
print("Features per sample :", X_train.shape[1])

print("\nU range :")
print(U_train.min(), "to", U_train.max())

print("\nV range :")
print(V_train.min(), "to", V_train.max())

print("\nTime :", round(elapsed, 2), "seconds")

print("\n===== STEP 9 COMPLETE =====")

In [ ]:
# ============================================
# STEP 10 — SCALE IMPROVED FEATURES
# ============================================

from sklearn.preprocessing import StandardScaler

print("===== STEP 10 — FEATURE SCALING =====")

feature_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(
    X_train
)

print("Original shape :", X_train.shape)
print("Scaled shape   :", X_train_scaled.shape)

print("\nMean of first 5 scaled features:")
print(
    np.mean(
        X_train_scaled[:, :5],
        axis=0
    )
)

print("\nStd of first 5 scaled features:")
print(
    np.std(
        X_train_scaled[:, :5],
        axis=0
    )
)

print("\n===== STEP 10 COMPLETE =====")

In [ ]:
# ============================================
# STEP 11 — BUILD IMPROVED VALIDATION DATASET
# ============================================

import time

print("===== STEP 11 — BUILDING VALIDATION DATA =====")

start_time = time.time()

X_val = []
U_val = []
V_val = []

for i, image_path in enumerate(val_images):

    X_img, U_img, V_img, _ = extract_image_training_data(
        image_path,
        image_width=400,
        n_segments=100
    )

    X_val.append(X_img)
    U_val.append(U_img)
    V_val.append(V_img)

    if (i + 1) % 25 == 0:
        print(
            f"Processed {i + 1}/{len(val_images)} validation images"
        )

# Combine all validation images
X_val = np.vstack(X_val)
U_val = np.concatenate(U_val)
V_val = np.concatenate(V_val)

# IMPORTANT:
# Use the scaler fitted on training data.
# Do NOT fit another scaler here.

X_val_scaled = feature_scaler.transform(
    X_val
)

elapsed = time.time() - start_time

print("\n===== VALIDATION DATA COMPLETE =====")

print("X_val shape        :", X_val.shape)
print("X_val_scaled shape:", X_val_scaled.shape)
print("U_val shape        :", U_val.shape)
print("V_val shape        :", V_val.shape)

print("\nValidation samples :", X_val.shape[0])
print("Features per sample :", X_val.shape[1])

print("\nU range :")
print(U_val.min(), "to", U_val.max())

print("\nV range :")
print(V_val.min(), "to", V_val.max())

print("\nTime :", round(elapsed, 2), "seconds")

print("\n===== STEP 11 COMPLETE =====")

In [ ]:
# ============================================
# STEP 12 — TRAIN IMPROVED SVR-U
# ============================================

from sklearn.svm import SVR
import time

print("===== STEP 12 — TRAINING IMPROVED SVR-U =====")

start_time = time.time()

svr_U = SVR(
    kernel="rbf",
    C=10,
    gamma="scale",
    epsilon=0.01
)

svr_U.fit(
    X_train_scaled,
    U_train
)

elapsed = time.time() - start_time

print("\n===== IMPROVED SVR-U TRAINING COMPLETE =====")

print("Kernel :", svr_U.kernel)
print("C :", svr_U.C)
print("Gamma :", svr_U.gamma)
print("Epsilon :", svr_U.epsilon)

print("Training samples :", X_train_scaled.shape[0])
print("Features :", X_train_scaled.shape[1])

print("Support vectors :", len(svr_U.support_))

print(
    "Training time :",
    round(elapsed, 2),
    "seconds"
)

print(
    "Training time :",
    round(elapsed / 60, 2),
    "minutes"
)

print("\n===== STEP 12 COMPLETE =====")

In [ ]:
from sklearn.svm import SVR
import time

print("===== STEP 13 — TRAINING IMPROVED SVR-V =====")

start_time = time.time()

svr_V = SVR(
    kernel="rbf",
    C=10,
    gamma="scale",
    epsilon=0.01
)

svr_V.fit(
    X_train_scaled,
    V_train
)

elapsed = time.time() - start_time

print("\n===== IMPROVED SVR-V TRAINING COMPLETE =====")
print("Kernel :", svr_V.kernel)
print("C :", svr_V.C)
print("Gamma :", svr_V.gamma)
print("Epsilon :", svr_V.epsilon)
print("Training samples :", X_train_scaled.shape[0])
print("Features :", X_train_scaled.shape[1])
print("Support vectors :", len(svr_V.support_))
print("Training time :", round(elapsed, 2), "seconds")
print("Training time :", round(elapsed / 60, 2), "minutes")

print("\n===== STEP 13 COMPLETE =====")

In [ ]:
import time
import numpy as np

print("===== STEP 14 — VALIDATION PREDICTION =====")

start_time = time.time()

# Predict chrominance channels
U_val_pred = svr_U.predict(X_val_scaled)
V_val_pred = svr_V.predict(X_val_scaled)

elapsed = time.time() - start_time

print("\n===== PREDICTION COMPLETE =====")
print("U predictions shape :", U_val_pred.shape)
print("V predictions shape :", V_val_pred.shape)

print("\nU predicted range :", U_val_pred.min(), "to", U_val_pred.max())
print("U actual range    :", U_val.min(), "to", U_val.max())

print("\nV predicted range :", V_val_pred.min(), "to", V_val_pred.max())
print("V actual range    :", V_val.min(), "to", V_val.max())

print("\nPrediction time :", round(elapsed, 2), "seconds")
print("Prediction time :", round(elapsed / 60, 2), "minutes")

print("\n===== STEP 14 COMPLETE =====")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("===== STEP 15 — VALIDATION ERROR =====")

# U channel errors
U_mae = mean_absolute_error(U_val, U_val_pred)
U_mse = mean_squared_error(U_val, U_val_pred)
U_rmse = np.sqrt(U_mse)

# V channel errors
V_mae = mean_absolute_error(V_val, V_val_pred)
V_mse = mean_squared_error(V_val, V_val_pred)
V_rmse = np.sqrt(V_mse)

print("\n===== U CHANNEL =====")
print("MAE  :", round(U_mae, 6))
print("MSE  :", round(U_mse, 6))
print("RMSE :", round(U_rmse, 6))

print("\n===== V CHANNEL =====")
print("MAE  :", round(V_mae, 6))
print("MSE  :", round(V_mse, 6))
print("RMSE :", round(V_rmse, 6))

print("\n===== AVERAGE =====")
print("Average MAE  :", round((U_mae + V_mae) / 2, 6))
print("Average RMSE :", round((U_rmse + V_rmse) / 2, 6))

print("\n===== STEP 15 COMPLETE =====")

In [ ]:
print("===== STEP 16 — COLORIZE ANOTHER VALIDATION IMAGE =====")

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import slic

# Use a DIFFERENT validation image
image_path = val_images[10]

# Load and resize
image_bgr = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

h, w = image_rgb.shape[:2]
new_w = 400
new_h = int(h * new_w / w)

image_rgb = cv2.resize(
    image_rgb,
    (new_w, new_h),
    interpolation=cv2.INTER_AREA
)

# RGB → YUV
image_float = image_rgb.astype(np.float32) / 255.0
image_yuv = cv2.cvtColor(image_float, cv2.COLOR_RGB2YUV)

Y = image_yuv[:, :, 0]

# SLIC
segments = slic(
    Y,
    n_segments=N_SEGMENTS,
    compactness=10,
    start_label=0,
    channel_axis=None
)

segment_ids = np.unique(segments)

# Extract 236 features
feature_result = extract_image_training_data(
    image_path,
    image_width=400,
    n_segments=N_SEGMENTS
)

X_image = feature_result[0]

# Scale using training scaler
X_image_scaled = feature_scaler.transform(X_image)

# Predict U and V
U_pred = svr_U.predict(X_image_scaled)
V_pred = svr_V.predict(X_image_scaled)

# Create U/V maps
U_map = np.zeros_like(Y, dtype=np.float32)
V_map = np.zeros_like(Y, dtype=np.float32)

for i, segment_id in enumerate(segment_ids):
    mask = segments == segment_id
    U_map[mask] = U_pred[i]
    V_map[mask] = V_pred[i]

# YUV → RGB
pred_yuv = np.stack([Y, U_map, V_map], axis=2)

pred_rgb = cv2.cvtColor(
    pred_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

pred_rgb = np.clip(pred_rgb, 0, 1)

# Display
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(Y, cmap="gray")
plt.title("Grayscale Input")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred_rgb)
plt.title("SVR Colorization")
plt.axis("off")

plt.tight_layout()
plt.show()

print("\nImage :", image_path)
print("Image size :", image_rgb.shape)
print("Segments :", len(segment_ids))
print("Features :", X_image.shape)

print("\n===== STEP 16 COMPLETE =====")

In [ ]:
print("===== STEP 17 — BUILD SLIC NEIGHBOR GRAPH =====")

import numpy as np

def build_neighbor_graph(segments):
    """
    Build an adjacency graph between neighboring SLIC segments.
    """

    segment_ids = np.unique(segments)
    segment_to_index = {
        segment_id: i
        for i, segment_id in enumerate(segment_ids)
    }

    neighbors = {
        i: set()
        for i in range(len(segment_ids))
    }

    # Horizontal boundaries
    left = segments[:, :-1]
    right = segments[:, 1:]

    mask = left != right

    for a, b in zip(left[mask], right[mask]):
        ia = segment_to_index[a]
        ib = segment_to_index[b]

        neighbors[ia].add(ib)
        neighbors[ib].add(ia)

    # Vertical boundaries
    top = segments[:-1, :]
    bottom = segments[1:, :]

    mask = top != bottom

    for a, b in zip(top[mask], bottom[mask]):
        ia = segment_to_index[a]
        ib = segment_to_index[b]

        neighbors[ia].add(ib)
        neighbors[ib].add(ia)

    return segment_ids, neighbors


segment_ids, neighbors = build_neighbor_graph(segments)

total_connections = sum(
    len(v) for v in neighbors.values()
) // 2

neighbor_counts = [
    len(v)
    for v in neighbors.values()
]

print("\nNumber of segments :", len(segment_ids))
print("Neighbor connections :", total_connections)
print("Minimum neighbors :", min(neighbor_counts))
print("Maximum neighbors :", max(neighbor_counts))
print("Average neighbors :", round(np.mean(neighbor_counts), 2))

print("\n===== STEP 17 COMPLETE =====")

In [ ]:
print("===== STEP 18 — MRF / ICM REFINEMENT =====")

import numpy as np
import time

MRF_GAMMA = 0.5
MRF_SIGMA2 = 0.01
MRF_ITERATIONS = 5

# Start from SVR predictions
U_mrf = U_pred.copy()
V_mrf = V_pred.copy()

# Segment mean luminance
segment_mean_Y = np.zeros(len(segment_ids), dtype=np.float32)

for i, segment_id in enumerate(segment_ids):
    mask = segments == segment_id
    segment_mean_Y[i] = np.mean(Y[mask])

# Candidate adjustments
candidate_offsets = np.array(
    [-0.08, -0.05, -0.03, -0.015, 0.0, 0.015, 0.03, 0.05, 0.08],
    dtype=np.float32
)

start_time = time.time()

for iteration in range(MRF_ITERATIONS):

    changes = 0

    for i in range(len(segment_ids)):

        current_u = U_mrf[i]
        current_v = V_mrf[i]

        best_u = current_u
        best_v = current_v
        best_energy = float("inf")

        # Generate candidate U/V values
        for du in candidate_offsets:
            for dv in candidate_offsets:

                candidate_u = current_u + du
                candidate_v = current_v + dv

                # Keep values in valid YUV range
                candidate_u = np.clip(candidate_u, 0, 1)
                candidate_v = np.clip(candidate_v, 0, 1)

                # Local energy:
                # stay reasonably close to original SVR prediction
                local_energy = (
                    (candidate_u - U_pred[i]) ** 2 +
                    (candidate_v - V_pred[i]) ** 2
                )

                neighbor_energy = 0.0

                # Compare with neighboring segments
                for j in neighbors[i]:

                    # Luminance similarity
                    dy = (
                        segment_mean_Y[i]
                        - segment_mean_Y[j]
                    )

                    weight = np.exp(
                        -(dy ** 2) /
                        (2 * MRF_SIGMA2)
                    )

                    # Encourage neighboring regions
                    # to have compatible chrominance
                    neighbor_energy += weight * (
                        (candidate_u - U_mrf[j]) ** 2 +
                        (candidate_v - V_mrf[j]) ** 2
                    )

                total_energy = (
                    local_energy +
                    MRF_GAMMA * neighbor_energy
                )

                if total_energy < best_energy:
                    best_energy = total_energy
                    best_u = candidate_u
                    best_v = candidate_v

        if (
            abs(best_u - U_mrf[i]) > 1e-6 or
            abs(best_v - V_mrf[i]) > 1e-6
        ):
            changes += 1

        U_mrf[i] = best_u
        V_mrf[i] = best_v

    print(
        "Iteration",
        iteration + 1,
        "changed segments:",
        changes
    )

elapsed = time.time() - start_time

print("\n===== MRF COMPLETE =====")
print("MRF Gamma :", MRF_GAMMA)
print("MRF Sigma² :", MRF_SIGMA2)
print("Iterations :", MRF_ITERATIONS)
print("Execution time :", round(elapsed, 2), "seconds")

print("\nSVR U range :",
      round(U_pred.min(), 4), "to",
      round(U_pred.max(), 4))

print("MRF U range :",
      round(U_mrf.min(), 4), "to",
      round(U_mrf.max(), 4))

print("\nSVR V range :",
      round(V_pred.min(), 4), "to",
      round(V_pred.max(), 4))

print("MRF V range :",
      round(V_mrf.min(), 4), "to",
      round(V_mrf.max(), 4))

print("\n===== STEP 18 COMPLETE =====")

In [ ]:
print("===== STEP 19 — SVR vs MRF COMPARISON =====")

# Create SVR U/V maps
U_svr_map = np.zeros_like(Y, dtype=np.float32)
V_svr_map = np.zeros_like(Y, dtype=np.float32)

# Create MRF U/V maps
U_mrf_map = np.zeros_like(Y, dtype=np.float32)
V_mrf_map = np.zeros_like(Y, dtype=np.float32)

for i, segment_id in enumerate(segment_ids):
    mask = segments == segment_id

    U_svr_map[mask] = U_pred[i]
    V_svr_map[mask] = V_pred[i]

    U_mrf_map[mask] = U_mrf[i]
    V_mrf_map[mask] = V_mrf[i]

# SVR-only color image
svr_yuv = np.stack(
    [Y, U_svr_map, V_svr_map],
    axis=2
)

svr_rgb = cv2.cvtColor(
    svr_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

svr_rgb = np.clip(svr_rgb, 0, 1)

# SVR + MRF color image
mrf_yuv = np.stack(
    [Y, U_mrf_map, V_mrf_map],
    axis=2
)

mrf_rgb = cv2.cvtColor(
    mrf_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

mrf_rgb = np.clip(mrf_rgb, 0, 1)

# Display comparison
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(Y, cmap="gray")
plt.title("Grayscale")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(svr_rgb)
plt.title("SVR Only")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(mrf_rgb)
plt.title("SVR + MRF")
plt.axis("off")

plt.tight_layout()
plt.show()

print("\n===== STEP 19 COMPLETE =====")

In [ ]:
print("===== STEP 20 — SVR vs MRF METRICS =====")

# --------------------------------------------------
# 1. Ground-truth U and V for this image
# --------------------------------------------------

true_U_map = U
true_V_map = V

true_U_segment = []
true_V_segment = []

for segment_id in segment_ids:
    mask = segments == segment_id

    true_U_segment.append(np.mean(true_U_map[mask]))
    true_V_segment.append(np.mean(true_V_map[mask]))

true_U_segment = np.array(true_U_segment)
true_V_segment = np.array(true_V_segment)


# --------------------------------------------------
# 2. U/V MAE
# --------------------------------------------------

svr_U_mae = np.mean(np.abs(true_U_segment - U_pred))
svr_V_mae = np.mean(np.abs(true_V_segment - V_pred))

mrf_U_mae = np.mean(np.abs(true_U_segment - U_mrf))
mrf_V_mae = np.mean(np.abs(true_V_segment - V_mrf))


# --------------------------------------------------
# 3. RGB error
# --------------------------------------------------

# Ground-truth RGB
true_rgb = image_rgb.astype(np.float32)

# SVR RGB error
svr_rgb_error = np.mean(
    np.abs(true_rgb - svr_rgb)
)

# MRF RGB error
mrf_rgb_error = np.mean(
    np.abs(true_rgb - mrf_rgb)
)


# --------------------------------------------------
# 4. Print results
# --------------------------------------------------

print("\n--- Chrominance Error ---")

print(f"SVR U MAE : {svr_U_mae:.6f}")
print(f"MRF U MAE : {mrf_U_mae:.6f}")

print(f"SVR V MAE : {svr_V_mae:.6f}")
print(f"MRF V MAE : {mrf_V_mae:.6f}")

print("\n--- RGB Error ---")

print(f"SVR RGB MAE : {svr_rgb_error:.6f}")
print(f"MRF RGB MAE : {mrf_rgb_error:.6f}")


# --------------------------------------------------
# 5. Improvement percentage
# --------------------------------------------------

rgb_improvement = (
    (svr_rgb_error - mrf_rgb_error)
    / svr_rgb_error
) * 100

print("\n--- MRF Improvement ---")
print(f"RGB error improvement: {rgb_improvement:.2f}%")

print("\n===== STEP 20 COMPLETE =====")

In [ ]:
# ============================================
# STEP 21 — DIAGNOSE U/V PREDICTION ERROR
# ============================================

print("===== STEP 21 — U/V ERROR DIAGNOSIS =====")

# ------------------------------------------------
# 1. Extract the TRUE U and V values
#    from the original second validation image
# ------------------------------------------------

original_yuv_second = image_yuv_second

U_true_second = []
V_true_second = []

for segment_id in segment_ids_second:

    mask = (segments_second == segment_id)

    U_true_second.append(
        np.mean(original_yuv_second[:, :, 1][mask])
    )

    V_true_second.append(
        np.mean(original_yuv_second[:, :, 2][mask])
    )

U_true_second = np.array(U_true_second)
V_true_second = np.array(V_true_second)

# ------------------------------------------------
# 2. Calculate prediction errors
# ------------------------------------------------

U_error_second = np.abs(
    U_true_second - U_second_svr
)

V_error_second = np.abs(
    V_true_second - V_second_svr
)

# ------------------------------------------------
# 3. Print overall errors
# ------------------------------------------------

print("\n----- U CHANNEL -----")

print("True U range:")
print(
    U_true_second.min(),
    "to",
    U_true_second.max()
)

print("Predicted U range:")
print(
    U_second_svr.min(),
    "to",
    U_second_svr.max()
)

print("Mean absolute U error:")
print(
    np.mean(U_error_second)
)

print("Maximum U error:")
print(
    np.max(U_error_second)
)

print("\n----- V CHANNEL -----")

print("True V range:")
print(
    V_true_second.min(),
    "to",
    V_true_second.max()
)

print("Predicted V range:")
print(
    V_second_svr.min(),
    "to",
    V_second_svr.max()
)

print("Mean absolute V error:")
print(
    np.mean(V_error_second)
)

print("Maximum V error:")
print(
    np.max(V_error_second)
)

# ------------------------------------------------
# 4. Find the worst-predicted segments
# ------------------------------------------------

worst_U = np.argsort(U_error_second)[-10:][::-1]
worst_V = np.argsort(V_error_second)[-10:][::-1]

print("\n===== WORST U SEGMENTS =====")

for i in worst_U:

    print(
        "Segment:", i,
        "| True:", round(U_true_second[i], 4),
        "| Pred:", round(U_second_svr[i], 4),
        "| Error:", round(U_error_second[i], 4)
    )

print("\n===== WORST V SEGMENTS =====")

for i in worst_V:

    print(
        "Segment:", i,
        "| True:", round(V_true_second[i], 4),
        "| Pred:", round(V_second_svr[i], 4),
        "| Error:", round(V_error_second[i], 4)
    )

print("\n===== STEP 21 COMPLETE =====")

In [ ]:
print("===== STEP 21 — SAVE CURRENT RESULT =====")

RESULT_DIR = "/content/colorization_results"

os.makedirs(RESULT_DIR, exist_ok=True)

# Convert images to uint8
original_uint8 = (np.clip(image_rgb, 0, 1) * 255).astype(np.uint8)
svr_uint8 = (np.clip(svr_rgb, 0, 1) * 255).astype(np.uint8)
mrf_uint8 = (np.clip(mrf_rgb, 0, 1) * 255).astype(np.uint8)

# Save individual results
cv2.imwrite(
    os.path.join(RESULT_DIR, "original.png"),
    cv2.cvtColor(original_uint8, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    os.path.join(RESULT_DIR, "svr_colorized.png"),
    cv2.cvtColor(svr_uint8, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    os.path.join(RESULT_DIR, "mrf_colorized.png"),
    cv2.cvtColor(mrf_uint8, cv2.COLOR_RGB2BGR)
)

# Create comparison image
comparison = np.hstack([
    original_uint8,
    np.repeat((Y[..., None] * 255).astype(np.uint8), 3, axis=2),
    svr_uint8,
    mrf_uint8
])

comparison_path = os.path.join(
    RESULT_DIR,
    "comparison_original_gray_svr_mrf.png"
)

cv2.imwrite(
    comparison_path,
    cv2.cvtColor(comparison, cv2.COLOR_RGB2BGR)
)

print("Results saved to:")
print(RESULT_DIR)

print("\nSaved files:")
print("1. original.png")
print("2. svr_colorized.png")
print("3. mrf_colorized.png")
print("4. comparison_original_gray_svr_mrf.png")

print("\n===== STEP 21 COMPLETE =====")

In [ ]:
print("===== STEP 22 — FULL VALIDATION EVALUATION =====")

# --------------------------------------------------
# Overall SVR validation metrics
# --------------------------------------------------

svr_U_mae = np.mean(np.abs(U_val - U_val_pred))
svr_U_rmse = np.sqrt(np.mean((U_val - U_val_pred) ** 2))

svr_V_mae = np.mean(np.abs(V_val - V_val_pred))
svr_V_rmse = np.sqrt(np.mean((V_val - V_val_pred) ** 2))

avg_mae = (svr_U_mae + svr_V_mae) / 2
avg_rmse = (svr_U_rmse + svr_V_rmse) / 2


print("\n--- FULL VALIDATION RESULTS ---")

print(f"Validation samples : {len(U_val)}")

print(f"\nU MAE  : {svr_U_mae:.6f}")
print(f"U RMSE : {svr_U_rmse:.6f}")

print(f"\nV MAE  : {svr_V_mae:.6f}")
print(f"V RMSE : {svr_V_rmse:.6f}")

print(f"\nAverage MAE  : {avg_mae:.6f}")
print(f"Average RMSE : {avg_rmse:.6f}")


# --------------------------------------------------
# Compare against simple grayscale baseline
#
# For grayscale baseline:
# U = mean U of training data
# V = mean V of training data
# --------------------------------------------------

baseline_U = np.mean(U_train)
baseline_V = np.mean(V_train)

baseline_U_pred = np.full_like(U_val, baseline_U)
baseline_V_pred = np.full_like(V_val, baseline_V)

baseline_U_mae = np.mean(
    np.abs(U_val - baseline_U_pred)
)

baseline_V_mae = np.mean(
    np.abs(V_val - baseline_V_pred)
)

baseline_avg_mae = (
    baseline_U_mae + baseline_V_mae
) / 2


print("\n--- BASELINE COMPARISON ---")

print(f"Baseline U MAE : {baseline_U_mae:.6f}")
print(f"SVR U MAE      : {svr_U_mae:.6f}")

print(f"\nBaseline V MAE : {baseline_V_mae:.6f}")
print(f"SVR V MAE      : {svr_V_mae:.6f}")

print(f"\nBaseline Avg MAE : {baseline_avg_mae:.6f}")
print(f"SVR Avg MAE      : {avg_mae:.6f}")


# --------------------------------------------------
# Improvement over baseline
# --------------------------------------------------

baseline_improvement = (
    (baseline_avg_mae - avg_mae)
    / baseline_avg_mae
) * 100

print(
    f"\nSVR improvement over baseline: "
    f"{baseline_improvement:.2f}%"
)

print("\n===== STEP 22 COMPLETE =====")

In [ ]:
print("===== STEP 23 — DEMO ON OUR IMAGE =====")

from google.colab import files

# Upload our image
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

print("\nImage selected:", image_path)

# --------------------------------------------------
# Load image
# --------------------------------------------------

img_bgr = cv2.imread(image_path)

if img_bgr is None:
    raise ValueError("Could not read the uploaded image.")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Resize
height, width = img_rgb.shape[:2]

new_width = 400
new_height = int(height * new_width / width)

image_rgb_demo = cv2.resize(
    img_rgb,
    (new_width, new_height)
)

# Normalize
image_rgb_demo = image_rgb_demo.astype(np.float32) / 255.0

# --------------------------------------------------
# Convert to YUV
# --------------------------------------------------

image_yuv_demo = cv2.cvtColor(
    image_rgb_demo,
    cv2.COLOR_RGB2YUV
)

Y_demo = image_yuv_demo[:, :, 0]

# --------------------------------------------------
# SLIC segmentation
# --------------------------------------------------

segments_demo = slic(
    Y_demo,
    n_segments=N_SEGMENTS,
    compactness=10,
    start_label=0,
    channel_axis=None
)

segment_ids_demo = np.unique(segments_demo)

print("Image size:", image_rgb_demo.shape)
print("Number of segments:", len(segment_ids_demo))

# --------------------------------------------------
# Extract features
# --------------------------------------------------

feature_result_demo = extract_image_training_data(
    image_path,
    image_width=400,
    n_segments=N_SEGMENTS
)

X_demo = feature_result_demo[0]

# Scale using TRAINING scaler
X_demo_scaled = feature_scaler.transform(X_demo)

print("Feature shape:", X_demo_scaled.shape)

# --------------------------------------------------
# SVR prediction
# --------------------------------------------------

U_demo_pred = svr_U.predict(X_demo_scaled)
V_demo_pred = svr_V.predict(X_demo_scaled)

print("SVR prediction complete.")

# --------------------------------------------------
# Create SVR maps
# --------------------------------------------------

U_svr_demo = np.zeros_like(Y_demo, dtype=np.float32)
V_svr_demo = np.zeros_like(Y_demo, dtype=np.float32)

for i, segment_id in enumerate(segment_ids_demo):

    mask = segments_demo == segment_id

    U_svr_demo[mask] = U_demo_pred[i]
    V_svr_demo[mask] = V_demo_pred[i]

# --------------------------------------------------
# SVR colorization
# --------------------------------------------------

svr_yuv_demo = np.stack(
    [Y_demo, U_svr_demo, V_svr_demo],
    axis=2
)

svr_rgb_demo = cv2.cvtColor(
    svr_yuv_demo.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

svr_rgb_demo = np.clip(svr_rgb_demo, 0, 1)

# --------------------------------------------------
# Build neighbor graph
# --------------------------------------------------

neighbors_demo = {
    int(segment_id): set()
    for segment_id in segment_ids_demo
}

for y in range(segments_demo.shape[0] - 1):

    for x in range(segments_demo.shape[1] - 1):

        current = int(segments_demo[y, x])
        right = int(segments_demo[y, x + 1])
        down = int(segments_demo[y + 1, x])

        if current != right:
            neighbors_demo[current].add(right)
            neighbors_demo[right].add(current)

        if current != down:
            neighbors_demo[current].add(down)
            neighbors_demo[down].add(current)

# --------------------------------------------------
# MRF / ICM
# --------------------------------------------------

U_mrf_demo = U_demo_pred.copy()
V_mrf_demo = V_demo_pred.copy()

segment_mean_Y_demo = np.zeros(len(segment_ids_demo))

for i, segment_id in enumerate(segment_ids_demo):

    mask = segments_demo == segment_id

    segment_mean_Y_demo[i] = np.mean(Y_demo[mask])

segment_index_demo = {
    int(segment_id): i
    for i, segment_id in enumerate(segment_ids_demo)
}

candidate_offsets = [
    -0.08, -0.05, -0.03, -0.015,
     0.0,
     0.015, 0.03, 0.05, 0.08
]

for iteration in range(MRF_ITERATIONS):

    changed = 0

    for i, segment_id in enumerate(segment_ids_demo):

        neighbor_indices = [
            segment_index_demo[n]
            for n in neighbors_demo[int(segment_id)]
        ]

        best_u = U_mrf_demo[i]
        best_v = V_mrf_demo[i]

        best_energy = float("inf")

        for du in candidate_offsets:

            for dv in candidate_offsets:

                candidate_u = np.clip(
                    U_mrf_demo[i] + du,
                    0,
                    1
                )

                candidate_v = np.clip(
                    V_mrf_demo[i] + dv,
                    0,
                    1
                )

                # Local energy
                local_energy = (
                    (candidate_u - U_demo_pred[i]) ** 2
                    +
                    (candidate_v - V_demo_pred[i]) ** 2
                )

                # Neighbor energy
                neighbor_energy = 0.0

                for j in neighbor_indices:

                    dy = (
                        segment_mean_Y_demo[i]
                        -
                        segment_mean_Y_demo[j]
                    )

                    weight = np.exp(
                        -(dy ** 2)
                        /
                        (2 * MRF_SIGMA2)
                    )

                    neighbor_energy += weight * (
                        (candidate_u - U_mrf_demo[j]) ** 2
                        +
                        (candidate_v - V_mrf_demo[j]) ** 2
                    )

                total_energy = (
                    local_energy
                    +
                    MRF_GAMMA * neighbor_energy
                )

                if total_energy < best_energy:

                    best_energy = total_energy
                    best_u = candidate_u
                    best_v = candidate_v

        if (
            abs(best_u - U_mrf_demo[i]) > 1e-8
            or
            abs(best_v - V_mrf_demo[i]) > 1e-8
        ):
            changed += 1

        U_mrf_demo[i] = best_u
        V_mrf_demo[i] = best_v

    print(
        f"MRF iteration {iteration + 1}: "
        f"{changed} segments changed"
    )

# --------------------------------------------------
# Create MRF maps
# --------------------------------------------------

U_mrf_map_demo = np.zeros_like(Y_demo)
V_mrf_map_demo = np.zeros_like(Y_demo)

for i, segment_id in enumerate(segment_ids_demo):

    mask = segments_demo == segment_id

    U_mrf_map_demo[mask] = U_mrf_demo[i]
    V_mrf_map_demo[mask] = V_mrf_demo[i]

# --------------------------------------------------
# Final SVR + MRF image
# --------------------------------------------------

mrf_yuv_demo = np.stack(
    [Y_demo, U_mrf_map_demo, V_mrf_map_demo],
    axis=2
)

mrf_rgb_demo = cv2.cvtColor(
    mrf_yuv_demo.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

mrf_rgb_demo = np.clip(mrf_rgb_demo, 0, 1)

# --------------------------------------------------
# Display final demo
# --------------------------------------------------

plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb_demo)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(Y_demo, cmap="gray")
plt.title("Grayscale Input")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(svr_rgb_demo)
plt.title("SVR Only")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(mrf_rgb_demo)
plt.title("SVR + MRF")
plt.axis("off")

plt.tight_layout()
plt.show()

print("\n===== STEP 23 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 24 — SAVE VERSION 1 MODEL
# ============================================================

import joblib
import os
import json

VERSION1_DIR = "/content/version1_svr_mrf"

os.makedirs(VERSION1_DIR, exist_ok=True)

# Save trained SVR models
joblib.dump(
    svr_U,
    os.path.join(VERSION1_DIR, "svr_U.pkl")
)

joblib.dump(
    svr_V,
    os.path.join(VERSION1_DIR, "svr_V.pkl")
)

# Save feature scaler
joblib.dump(
    feature_scaler,
    os.path.join(VERSION1_DIR, "feature_scaler.pkl")
)

# Save important project settings
settings = {
    "image_width": 400,
    "n_segments": 100,
    "feature_count": 236,

    "svr_kernel": "rbf",
    "svr_C": 10,
    "svr_gamma": "scale",
    "svr_epsilon": 0.01,

    "mrf_gamma": 0.5,
    "mrf_sigma2": 0.01,
    "mrf_iterations": 5,

    "validation_samples": 9600,

    "validation_U_MAE": float(svr_U_mae),
    "validation_U_RMSE": float(svr_U_rmse),

    "validation_V_MAE": float(svr_V_mae),
    "validation_V_RMSE": float(svr_V_rmse),

    "validation_average_MAE": float(avg_mae),
    "validation_average_RMSE": float(avg_rmse),

    "baseline_average_MAE": float(baseline_avg_mae)
}

with open(
    os.path.join(VERSION1_DIR, "version1_results.json"),
    "w"
) as f:
    json.dump(settings, f, indent=4)

print("===== VERSION 1 SAVED =====")
print(VERSION1_DIR)

print("\nSaved:")
print("1. svr_U.pkl")
print("2. svr_V.pkl")
print("3. feature_scaler.pkl")
print("4. version1_results.json")

In [ ]:
# ZIP Version 1
!zip -r /content/version1_svr_mrf.zip /content/version1_svr_mrf

In [ ]:
from google.colab import files

files.download("/content/version1_svr_mrf.zip")

In [ ]:
# ============================================================
# STEP 3 — LOAD V1 FEATURE DATA
# ============================================================

import numpy as np

NPZ_PATH = "/content/version1_feature_data.npz"

data = np.load(NPZ_PATH)

X_train = data["X_train"]
U_train = data["U_train"]
V_train = data["V_train"]

X_val = data["X_val"]
U_val = data["U_val"]
V_val = data["V_val"]

print("===== V1 FEATURE DATA LOADED =====")

print("X_train :", X_train.shape)
print("U_train :", U_train.shape)
print("V_train :", V_train.shape)

print("X_val   :", X_val.shape)
print("U_val   :", U_val.shape)
print("V_val   :", V_val.shape)

In [ ]:
# ============================================================
# STEP 4 — SELECT 20 USEFUL FEATURES
# ============================================================

import numpy as np

selected_features_20 = np.concatenate([
    np.arange(200, 204),   # Basic luminance statistics: 4
    np.arange(204, 209),   # Percentiles: 5
    np.arange(225, 228),   # Gradient features: 3
    np.arange(228, 230),   # Texture features: 2
    np.arange(230, 234),   # Multi-scale features: 4
    np.arange(234, 236)    # Spatial coordinates: 2
])

X_train_20 = X_train[:, selected_features_20]
X_val_20 = X_val[:, selected_features_20]

print("===== 20 FEATURE SELECTION =====")

print("Original features :", X_train.shape[1])
print("Selected features :", len(selected_features_20))

print("\nX_train_20:", X_train_20.shape)
print("X_val_20  :", X_val_20.shape)

print("\nSelected feature indices:")
print(selected_features_20)

In [ ]:
# ============================================================
# STEP 5 — ADD 4 LOCAL QUADRANT FEATURES
# ============================================================

# First 100 features of V1 = raw 10x10 luminance patch
raw_train_patches = X_train[:, :100].reshape(-1, 10, 10)
raw_val_patches   = X_val[:, :100].reshape(-1, 10, 10)

# ---- Training local features ----
train_top_left = np.mean(raw_train_patches[:, :5, :5], axis=(1, 2))
train_top_right = np.mean(raw_train_patches[:, :5, 5:], axis=(1, 2))
train_bottom_left = np.mean(raw_train_patches[:, 5:, :5], axis=(1, 2))
train_bottom_right = np.mean(raw_train_patches[:, 5:, 5:], axis=(1, 2))

local_train = np.column_stack([
    train_top_left,
    train_top_right,
    train_bottom_left,
    train_bottom_right
])

# ---- Validation local features ----
val_top_left = np.mean(raw_val_patches[:, :5, :5], axis=(1, 2))
val_top_right = np.mean(raw_val_patches[:, :5, 5:], axis=(1, 2))
val_bottom_left = np.mean(raw_val_patches[:, 5:, :5], axis=(1, 2))
val_bottom_right = np.mean(raw_val_patches[:, 5:, 5:], axis=(1, 2))

local_val = np.column_stack([
    val_top_left,
    val_top_right,
    val_bottom_left,
    val_bottom_right
])

# Combine 20 selected features + 4 new local features
X_train_24 = np.hstack([X_train_20, local_train])
X_val_24 = np.hstack([X_val_20, local_val])

print("===== 24 FEATURE DATA =====")

print("20 selected features :", X_train_20.shape[1])
print("4 local features     :", local_train.shape[1])
print("Total features       :", X_train_24.shape[1])

print("\nX_train_24:", X_train_24.shape)
print("X_val_24  :", X_val_24.shape)

In [ ]:
# ============================================================
# STEP 6 — SCALE FINAL 24 FEATURES
# ============================================================

from sklearn.preprocessing import StandardScaler

v2_final_scaler = StandardScaler()

# Learn scaling only from training data
X_train_scaled = v2_final_scaler.fit_transform(X_train_24)

# Apply the same scaling to validation data
X_val_scaled = v2_final_scaler.transform(X_val_24)

print("===== FEATURE SCALING =====")

print("Before scaling:")
print("X_train_24:", X_train_24.shape)
print("X_val_24  :", X_val_24.shape)

print("\nAfter scaling:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled  :", X_val_scaled.shape)

print("\nTraining mean (first 5 features):")
print(np.mean(X_train_scaled[:, :5], axis=0))

print("\nTraining std (first 5 features):")
print(np.std(X_train_scaled[:, :5], axis=0))

In [ ]:
# ============================================================
# STEP 7 — TRAIN FINAL SVR MODELS
# ============================================================

from sklearn.svm import SVR
import time

print("===== TRAINING FINAL SVR MODELS =====")

# ------------------------------------------------------------
# SVR for U chrominance
# ------------------------------------------------------------

start_time = time.time()

v2_final_svr_U = SVR(
    kernel="rbf",
    C=1,
    gamma=0.01,
    epsilon=0.01
)

v2_final_svr_U.fit(X_train_scaled, U_train)

U_train_time = time.time() - start_time

print(f"SVR-U training time: {U_train_time:.2f} seconds")
print(f"SVR-U support vectors: {len(v2_final_svr_U.support_)}")


# ------------------------------------------------------------
# SVR for V chrominance
# ------------------------------------------------------------

start_time = time.time()

v2_final_svr_V = SVR(
    kernel="rbf",
    C=1,
    gamma=0.01,
    epsilon=0.01
)

v2_final_svr_V.fit(X_train_scaled, V_train)

V_train_time = time.time() - start_time

print(f"SVR-V training time: {V_train_time:.2f} seconds")
print(f"SVR-V support vectors: {len(v2_final_svr_V.support_)}")


print("\n===== TRAINING COMPLETE =====")
print(f"Total training time: {U_train_time + V_train_time:.2f} seconds")

In [ ]:
# ============================================================
# STEP 8 — VALIDATION PREDICTION
# ============================================================

import time

print("===== VALIDATION PREDICTION =====")

start_time = time.time()

U_pred = v2_final_svr_U.predict(X_val_scaled)
V_pred = v2_final_svr_V.predict(X_val_scaled)

prediction_time = time.time() - start_time

print("U prediction shape:", U_pred.shape)
print("V prediction shape:", V_pred.shape)

print("\nU prediction range:")
print("Min:", U_pred.min())
print("Max:", U_pred.max())

print("\nV prediction range:")
print("Min:", V_pred.min())
print("Max:", V_pred.max())

print(f"\nPrediction time: {prediction_time:.2f} seconds")

In [ ]:
# ============================================================
# STEP 9 — FINAL VALIDATION MAE
# ============================================================

from sklearn.metrics import mean_absolute_error

# Calculate U and V errors
U_MAE = mean_absolute_error(U_val, U_pred)
V_MAE = mean_absolute_error(V_val, V_pred)

# Average of U and V MAE
Average_MAE = (U_MAE + V_MAE) / 2

# Previous reference values
baseline_mae = 0.038742
previous_v2_mae = 0.037046

# Improvement percentages
improvement_vs_baseline = (
    (baseline_mae - Average_MAE) / baseline_mae
) * 100

improvement_vs_previous_v2 = (
    (previous_v2_mae - Average_MAE) / previous_v2_mae
) * 100


print("===== FINAL VALIDATION RESULTS =====")

print(f"U MAE       : {U_MAE:.6f}")
print(f"V MAE       : {V_MAE:.6f}")
print(f"Average MAE : {Average_MAE:.6f}")

print("\nBaseline MAE:")
print(f"{baseline_mae:.6f}")

print(f"\nImprovement vs baseline:")
print(f"{improvement_vs_baseline:.2f}%")

print("\nPrevious V2 MAE:")
print(f"{previous_v2_mae:.6f}")

print(f"\nImprovement vs previous V2:")
print(f"{improvement_vs_previous_v2:.2f}%")

In [ ]:
# ============================================================
# STEP 10A — DOWNLOAD AND EXTRACT BSDS500
# ============================================================

import os
import urllib.request
import tarfile

URL = "https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz"

TGZ_PATH = "/content/BSR_bsds500.tgz"
EXTRACT_DIR = "/content"

print("===== DOWNLOADING BSDS500 =====")

if not os.path.exists(TGZ_PATH):
    urllib.request.urlretrieve(URL, TGZ_PATH)
    print("Download complete.")
else:
    print("Archive already exists.")

print("\n===== EXTRACTING BSDS500 =====")

with tarfile.open(TGZ_PATH, "r:gz") as tar:
    tar.extractall(EXTRACT_DIR)

print("Extraction complete.")

print("\n===== CHECK =====")
print(os.path.exists("/content/BSR"))

In [ ]:
# ============================================================
# STEP 10B — VERIFY BSDS500 PATHS
# ============================================================

import os

DATA_DIR = "/content/BSR/BSDS500"

TRAIN_DIR = os.path.join(DATA_DIR, "data", "images", "train")
VAL_DIR   = os.path.join(DATA_DIR, "data", "images", "val")
TEST_DIR  = os.path.join(DATA_DIR, "data", "images", "test")

def get_image_files(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

train_images = get_image_files(TRAIN_DIR)
val_images   = get_image_files(VAL_DIR)
test_images  = get_image_files(TEST_DIR)

print("===== BSDS500 PATHS =====")
print("Train      :", TRAIN_DIR)
print("Validation :", VAL_DIR)
print("Test       :", TEST_DIR)

print("\n===== IMAGE COUNTS =====")
print("Train      :", len(train_images))
print("Validation :", len(val_images))
print("Test       :", len(test_images))
print("Total      :", len(train_images) + len(val_images) + len(test_images))

print("\n===== STEP 10B COMPLETE =====")

In [ ]:
# ============================================================
# STEP 11 — PREPARE DEMO IMAGE + SLIC
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import slic, mark_boundaries

# Use the first validation image
demo_path = val_images[2]

# Read image
image_bgr = cv2.imread(demo_path)

# Convert BGR → RGB
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Convert to float [0, 1]
image_rgb = image_rgb.astype(np.float32) / 255.0

# Resize to width 400 while keeping aspect ratio
h, w = image_rgb.shape[:2]
new_width = 400
new_height = int(h * new_width / w)

image_rgb = cv2.resize(
    image_rgb,
    (new_width, new_height),
    interpolation=cv2.INTER_AREA
)

# RGB → YUV
image_yuv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2YUV)

Y_demo = image_yuv[:, :, 0]
U_demo = image_yuv[:, :, 1]
V_demo = image_yuv[:, :, 2]

# SLIC
N_SEGMENTS = 100

segments_demo = slic(
    image_rgb,
    n_segments=N_SEGMENTS,
    compactness=10,
    start_label=0
)

number_of_segments = len(np.unique(segments_demo))

print("===== DEMO IMAGE INFORMATION =====")
print("Image:", os.path.basename(demo_path))
print("Image shape:", image_rgb.shape)
print("Y shape:", Y_demo.shape)
print("Requested SLIC segments:", N_SEGMENTS)
print("Actual SLIC segments:", number_of_segments)

# Display SLIC
slic_view = mark_boundaries(
    image_rgb,
    segments_demo,
    color=(1, 1, 0),
    mode="outer"
)

plt.figure(figsize=(10, 6))
plt.imshow(slic_view)
plt.title(f"SLIC Superpixels — {number_of_segments} regions")
plt.axis("off")
plt.show()

print("\n===== STEP 11 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 12 — EXTRACT 236 FEATURES FOR DEMO IMAGE
# ============================================================

from skimage.filters import sobel
from scipy.ndimage import gaussian_filter, laplace

def extract_features_for_segment(Y, segments, segment_id):
    """
    Extract the same 236-feature representation used by V1.
    """

    coords = np.argwhere(segments == segment_id)

    # Segment centroid
    cy, cx = coords.mean(axis=0)

    # --------------------------------------------------------
    # 10x10 luminance patch
    # --------------------------------------------------------

    y_int = int(round(cy))
    x_int = int(round(cx))

    half = 5

    y1 = max(0, y_int - half)
    y2 = min(Y.shape[0], y_int + half)
    x1 = max(0, x_int - half)
    x2 = min(Y.shape[1], x_int + half)

    patch = Y[y1:y2, x1:x2]

    # Pad if patch is smaller than 10x10
    padded_patch = np.zeros((10, 10), dtype=np.float32)

    ph, pw = patch.shape
    padded_patch[:ph, :pw] = patch[:10, :10]

    raw_patch_features = padded_patch.flatten()

    # --------------------------------------------------------
    # FFT magnitude — 100 features
    # --------------------------------------------------------

    fft = np.fft.fft2(padded_patch)
    fft_magnitude = np.abs(fft)

    fft_magnitude = fft_magnitude / (
        np.max(fft_magnitude) + 1e-8
    )

    fft_features = fft_magnitude.flatten()

    # --------------------------------------------------------
    # Basic luminance statistics — 4
    # --------------------------------------------------------

    segment_values = Y[segments == segment_id]

    basic_features = np.array([
        np.mean(segment_values),
        np.std(segment_values),
        np.min(segment_values),
        np.max(segment_values)
    ])

    # --------------------------------------------------------
    # Percentiles — 5
    # --------------------------------------------------------

    percentile_features = np.percentile(
        segment_values,
        [10, 25, 50, 75, 90]
    )

    # --------------------------------------------------------
    # Histogram — 16
    # --------------------------------------------------------

    histogram, _ = np.histogram(
        segment_values,
        bins=16,
        range=(0, 1)
    )

    histogram = histogram.astype(np.float32)

    histogram = histogram / (
        np.sum(histogram) + 1e-8
    )

    # --------------------------------------------------------
    # Gradient features — 3
    # --------------------------------------------------------

    gradient_image = sobel(Y)

    gradient_values = gradient_image[
        segments == segment_id
    ]

    gradient_features = np.array([
        np.mean(gradient_values),
        np.std(gradient_values),
        np.max(gradient_values)
    ])

    # --------------------------------------------------------
    # Texture / Laplacian — 2
    # --------------------------------------------------------

    laplacian_image = laplace(Y)

    laplacian_values = laplacian_image[
        segments == segment_id
    ]

    texture_features = np.array([
        np.mean(np.abs(laplacian_values)),
        np.std(laplacian_values)
    ])

    # --------------------------------------------------------
    # Multi-scale features — 4
    # --------------------------------------------------------

    blur_5 = gaussian_filter(Y, sigma=5)
    blur_15 = gaussian_filter(Y, sigma=15)

    multi_scale_features = np.array([
        np.mean(blur_5[segments == segment_id]),
        np.std(blur_5[segments == segment_id]),
        np.mean(blur_15[segments == segment_id]),
        np.std(blur_15[segments == segment_id])
    ])

    # --------------------------------------------------------
    # Spatial coordinates — 2
    # --------------------------------------------------------

    spatial_features = np.array([
        cx / Y.shape[1],
        cy / Y.shape[0]
    ])

    # --------------------------------------------------------
    # Combine everything
    # --------------------------------------------------------

    features = np.concatenate([
        raw_patch_features,       # 100
        fft_features,             # 100
        basic_features,           # 4
        percentile_features,      # 5
        histogram,                # 16
        gradient_features,        # 3
        texture_features,         # 2
        multi_scale_features,     # 4
        spatial_features          # 2
    ])

    return features


# Extract features for every SLIC region
segment_ids = np.unique(segments_demo)

X_demo_236 = np.array([
    extract_features_for_segment(
        Y_demo,
        segments_demo,
        segment_id
    )
    for segment_id in segment_ids
])

print("===== DEMO FEATURE EXTRACTION =====")
print("Number of segments :", len(segment_ids))
print("Feature matrix     :", X_demo_236.shape)
print("Features per segment:", X_demo_236.shape[1])

print("\nExpected features  : 236")

print("\n===== STEP 12 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 13 — CREATE FINAL 24 DEMO FEATURES
# ============================================================

# ------------------------------------------------------------
# 1. Select the same 20 features used during training
# ------------------------------------------------------------

X_demo_20 = X_demo_236[:, selected_features_20]

# ------------------------------------------------------------
# 2. Get original 10x10 luminance patches
#    Features 0–99 = raw 10x10 patch
# ------------------------------------------------------------

raw_demo_patches = X_demo_236[:, :100].reshape(-1, 10, 10)

# ------------------------------------------------------------
# 3. Create 4 local quadrant features
# ------------------------------------------------------------

top_left = np.mean(
    raw_demo_patches[:, :5, :5],
    axis=(1, 2)
)

top_right = np.mean(
    raw_demo_patches[:, :5, 5:],
    axis=(1, 2)
)

bottom_left = np.mean(
    raw_demo_patches[:, 5:, :5],
    axis=(1, 2)
)

bottom_right = np.mean(
    raw_demo_patches[:, 5:, 5:],
    axis=(1, 2)
)

local_demo_features = np.column_stack([
    top_left,
    top_right,
    bottom_left,
    bottom_right
])

# ------------------------------------------------------------
# 4. Combine 20 + 4 = 24
# ------------------------------------------------------------

X_demo_24 = np.hstack([
    X_demo_20,
    local_demo_features
])

print("===== FINAL DEMO FEATURES =====")

print("Original features :", X_demo_236.shape[1])
print("Selected features :", X_demo_20.shape[1])
print("Local features    :", local_demo_features.shape[1])
print("Final features    :", X_demo_24.shape[1])

print("\nX_demo_20:", X_demo_20.shape)
print("Local features:", local_demo_features.shape)
print("X_demo_24:", X_demo_24.shape)

print("\n===== STEP 13 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 14 — SCALE DEMO FEATURES
# ============================================================

X_demo_scaled = v2_final_scaler.transform(X_demo_24)

print("===== DEMO FEATURE SCALING =====")
print("Before scaling :", X_demo_24.shape)
print("After scaling  :", X_demo_scaled.shape)

print("\n===== STEP 14 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 15 — PREDICT U AND V FOR DEMO IMAGE
# ============================================================

import time

print("===== DEMO SVR PREDICTION =====")

start_time = time.time()

# Predict chrominance
U_pred_demo = v2_final_svr_U.predict(X_demo_scaled)
V_pred_demo = v2_final_svr_V.predict(X_demo_scaled)

prediction_time = time.time() - start_time

print("U prediction shape:", U_pred_demo.shape)
print("V prediction shape:", V_pred_demo.shape)

print("\nU prediction range:")
print("Min:", U_pred_demo.min())
print("Max:", U_pred_demo.max())

print("\nV prediction range:")
print("Min:", V_pred_demo.min())
print("Max:", V_pred_demo.max())

print(f"\nPrediction time: {prediction_time:.2f} seconds")

print("\n===== STEP 15 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 16 — RECONSTRUCT SVR-ONLY COLORIZED IMAGE
# ============================================================

# Create empty U and V images
U_svr = np.zeros_like(Y_demo)
V_svr = np.zeros_like(Y_demo)

# Assign predicted U/V to each SLIC region
for i, segment_id in enumerate(segment_ids):
    mask = segments_demo == segment_id
    U_svr[mask] = U_pred_demo[i]
    V_svr[mask] = V_pred_demo[i]

# Combine Y + predicted U + predicted V
svr_yuv = np.stack([
    Y_demo,
    U_svr,
    V_svr
], axis=2)

# YUV → RGB
svr_rgb = cv2.cvtColor(
    svr_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

# Keep values valid
svr_rgb = np.clip(svr_rgb, 0, 1)

print("===== SVR-ONLY COLORIZATION =====")
print("SVR RGB shape:", svr_rgb.shape)
print("===== STEP 16 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 17 — BUILD SLIC NEIGHBOR GRAPH
# ============================================================

segment_ids = np.unique(segments_demo)

neighbors = {int(s): set() for s in segment_ids}

h, w = segments_demo.shape

for y in range(h):
    for x in range(w):
        current = int(segments_demo[y, x])

        if x + 1 < w:
            right = int(segments_demo[y, x + 1])
            if current != right:
                neighbors[current].add(right)
                neighbors[right].add(current)

        if y + 1 < h:
            bottom = int(segments_demo[y + 1, x])
            if current != bottom:
                neighbors[current].add(bottom)
                neighbors[bottom].add(current)

num_connections = sum(len(v) for v in neighbors.values()) // 2

print("Number of SLIC regions:", len(segment_ids))
print("Neighbor connections:", num_connections)
print("===== STEP 17 COMPLETE =====")

In [ ]:

# ============================================================
# STEP 18 — MRF COLOR SMOOTHING
# ============================================================

# Start with the original SVR predictions
U_mrf = U_pred_demo.copy()
V_mrf = V_pred_demo.copy()

# Gentler smoothing settings
MRF_GAMMA = 1
MRF_ITERATIONS = 3

# Map each superpixel ID to its array index
segment_to_index = {
    int(segment_id): i
    for i, segment_id in enumerate(segment_ids)
}

total_changes = 0

for iteration in range(MRF_ITERATIONS):

    changes = 0

    for i, segment_id in enumerate(segment_ids):

        # Find neighboring superpixels
        neighbour_ids = neighbors[int(segment_id)]

        if len(neighbour_ids) == 0:
            continue

        # Convert neighbor IDs to array indices
        neighbour_indices = [
            segment_to_index[int(n)]
            for n in neighbour_ids
            if int(n) in segment_to_index
        ]

        if len(neighbour_indices) == 0:
            continue

        # Calculate average color of neighboring superpixels
        mean_U = np.mean(U_mrf[neighbour_indices])
        mean_V = np.mean(V_mrf[neighbour_indices])

        # Store the current color values
        old_U = U_mrf[i]
        old_V = V_mrf[i]

        # Blend the original color with neighboring colors
        new_U = (
            old_U + MRF_GAMMA * mean_U
        ) / (1 + MRF_GAMMA)

        new_V = (
            old_V + MRF_GAMMA * mean_V
        ) / (1 + MRF_GAMMA)

        # Count meaningful updates
        if (
            abs(new_U - old_U) > 1e-6
            or abs(new_V - old_V) > 1e-6
        ):
            changes += 1

        # Update the color values
        U_mrf[i] = new_U
        V_mrf[i] = new_V

    total_changes += changes

    print(
        f"Iteration {iteration + 1}: "
        f"{changes} segment updates"
    )

print("Total updates:", total_changes)
print("===== STEP 18 COMPLETE =====")


In [ ]:
# ============================================================
# STEP 19 — RECONSTRUCT SVR + MRF IMAGE
# ============================================================

print("===== STEP 19 — CHECKING VARIABLES =====")

print("Y_demo shape      :", Y_demo.shape)
print("SLIC shape        :", segments_demo.shape)
print("Number of regions :", len(segment_ids))
print("U_mrf shape       :", U_mrf.shape)
print("V_mrf shape       :", V_mrf.shape)

# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

assert Y_demo.shape == segments_demo.shape, \
    "ERROR: Y_demo and segments_demo have different image sizes."

assert len(U_mrf) == len(segment_ids), \
    "ERROR: U_mrf and SLIC regions do not match."

assert len(V_mrf) == len(segment_ids), \
    "ERROR: V_mrf and SLIC regions do not match."

# ------------------------------------------------------------
# CREATE FULL-SIZE U/V IMAGES
# ------------------------------------------------------------

U_mrf_image = np.zeros_like(Y_demo, dtype=np.float32)
V_mrf_image = np.zeros_like(Y_demo, dtype=np.float32)

# Put each predicted MRF value back into its SLIC region
for i, segment_id in enumerate(segment_ids):

    mask = (segments_demo == segment_id)

    U_mrf_image[mask] = U_mrf[i]
    V_mrf_image[mask] = V_mrf[i]

# ------------------------------------------------------------
# COMBINE Y + MRF U + MRF V
# ------------------------------------------------------------

mrf_yuv = np.stack(
    [
        Y_demo,
        U_mrf_image,
        V_mrf_image
    ],
    axis=2
)

# ------------------------------------------------------------
# YUV → RGB
# ------------------------------------------------------------

mrf_rgb = cv2.cvtColor(
    mrf_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

mrf_rgb = np.clip(mrf_rgb, 0, 1)

print("\n===== SVR + MRF COLORIZATION =====")
print("MRF RGB shape:", mrf_rgb.shape)
print("===== STEP 19 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 20 — DISPLAY SVR + MRF RESULT
# ============================================================

plt.figure(figsize=(8, 6))

plt.imshow(mrf_rgb)

plt.title("SVR + MRF Colorization")
plt.axis("off")

plt.show()

print("===== STEP 20 COMPLETE =====")

In [ ]:
# ============================================================
# STEP 20 — FINAL 5-PANEL DEMO
# ============================================================

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

# 1. Original
axes[0].imshow(image_rgb)
axes[0].set_title("Original")

# 2. Grayscale
axes[1].imshow(Y_demo, cmap="gray")
axes[1].set_title("Grayscale")

# 3. SLIC
axes[2].imshow(slic_view)
axes[2].set_title("SLIC Superpixels")

# 4. SVR
axes[3].imshow(svr_rgb)
axes[3].set_title("SVR Prediction")

# 5. SVR + MRF
axes[4].imshow(mrf_rgb)
axes[4].set_title("SVR + MRF")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("===== FINAL DEMO COMPLETE =====")

In [ ]:
# ============================================================
# FINAL 5-PANEL DEMO — 101085.jpg
# ============================================================

# 1. Load image
image_bgr = cv2.imread(demo_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_rgb = image_rgb.astype(np.float32) / 255.0

# Resize
new_width = 400
new_height = int(image_rgb.shape[0] * new_width / image_rgb.shape[1])

image_rgb = cv2.resize(
    image_rgb,
    (new_width, new_height),
    interpolation=cv2.INTER_AREA
)

# 2. Convert to YUV
image_yuv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2YUV)

Y_demo = image_yuv[:, :, 0]

# 3. SLIC
segments_demo = slic(
    image_rgb,
    n_segments=100,
    compactness=10,
    start_label=0
)

segment_ids = np.unique(segments_demo)

slic_view = mark_boundaries(
    image_rgb,
    segments_demo,
    color=(1, 1, 0),
    mode="outer"
)

# ============================================================
# 4. EXTRACT 236 FEATURES
# ============================================================

X_demo_236 = []

for segment_id in segment_ids:

    features = extract_features_for_segment(
        Y_demo,
        segments_demo,
        segment_id
    )

    X_demo_236.append(features)

X_demo_236 = np.array(X_demo_236)

# ============================================================
# 5. CREATE 24 FEATURES
# ============================================================

X_demo_20 = X_demo_236[:, selected_features_20]

raw_demo_patches = X_demo_236[:, :100].reshape(-1, 10, 10)

top_left = np.mean(raw_demo_patches[:, :5, :5], axis=(1, 2))
top_right = np.mean(raw_demo_patches[:, :5, 5:], axis=(1, 2))
bottom_left = np.mean(raw_demo_patches[:, 5:, :5], axis=(1, 2))
bottom_right = np.mean(raw_demo_patches[:, 5:, 5:], axis=(1, 2))

local_demo_features = np.column_stack([
    top_left,
    top_right,
    bottom_left,
    bottom_right
])

X_demo_24 = np.hstack([
    X_demo_20,
    local_demo_features
])

# ============================================================
# 6. SCALE
# ============================================================

X_demo_scaled = v2_final_scaler.transform(X_demo_24)

# ============================================================
# 7. SVR PREDICTION
# ============================================================

U_pred_demo = v2_final_svr_U.predict(X_demo_scaled)
V_pred_demo = v2_final_svr_V.predict(X_demo_scaled)

# ============================================================
# 8. SVR RECONSTRUCTION
# ============================================================

U_svr = np.zeros_like(Y_demo)
V_svr = np.zeros_like(Y_demo)

for i, segment_id in enumerate(segment_ids):

    mask = segments_demo == segment_id

    U_svr[mask] = U_pred_demo[i]
    V_svr[mask] = V_pred_demo[i]

svr_yuv = np.stack([
    Y_demo,
    U_svr,
    V_svr
], axis=2)

svr_rgb = cv2.cvtColor(
    svr_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

svr_rgb = np.clip(svr_rgb, 0, 1)

# ============================================================
# 9. BUILD SLIC NEIGHBOR GRAPH
# ============================================================

neighbors = {
    int(s): set()
    for s in segment_ids
}

h, w = segments_demo.shape

for y in range(h):

    for x in range(w):

        current = int(segments_demo[y, x])

        if x + 1 < w:

            right = int(segments_demo[y, x + 1])

            if current != right:

                neighbors[current].add(right)
                neighbors[right].add(current)

        if y + 1 < h:

            bottom = int(segments_demo[y + 1, x])

            if current != bottom:

                neighbors[current].add(bottom)
                neighbors[bottom].add(current)

# ============================================================
# 10. MRF / ICM
# ============================================================

U_mrf = U_pred_demo.copy()
V_mrf = V_pred_demo.copy()

MRF_GAMMA = 5
MRF_ITERATIONS = 5

segment_to_index = {
    int(segment_id): i
    for i, segment_id in enumerate(segment_ids)
}

for iteration in range(MRF_ITERATIONS):

    for i, segment_id in enumerate(segment_ids):

        neighbour_ids = neighbors[int(segment_id)]

        if len(neighbour_ids) == 0:
            continue

        neighbour_indices = [
            segment_to_index[int(n)]
            for n in neighbour_ids
        ]

        mean_U = np.mean(U_mrf[neighbour_indices])
        mean_V = np.mean(V_mrf[neighbour_indices])

        U_mrf[i] = (
            U_mrf[i] + MRF_GAMMA * mean_U
        ) / (1 + MRF_GAMMA)

        V_mrf[i] = (
            V_mrf[i] + MRF_GAMMA * mean_V
        ) / (1 + MRF_GAMMA)

# ============================================================
# 11. MRF RECONSTRUCTION
# ============================================================

U_mrf_image = np.zeros_like(Y_demo, dtype=np.float32)
V_mrf_image = np.zeros_like(Y_demo, dtype=np.float32)

for i, segment_id in enumerate(segment_ids):

    mask = segments_demo == segment_id

    U_mrf_image[mask] = U_mrf[i]
    V_mrf_image[mask] = V_mrf[i]

mrf_yuv = np.stack([
    Y_demo,
    U_mrf_image,
    V_mrf_image
], axis=2)

mrf_rgb = cv2.cvtColor(
    mrf_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

mrf_rgb = np.clip(mrf_rgb, 0, 1)

# ============================================================
# 12. FINAL 5-PANEL DEMO
# ============================================================

fig, axes = plt.subplots(
    1, 5,
    figsize=(22, 5)
)

# Original
axes[0].imshow(image_rgb)
axes[0].set_title("Original")

# Grayscale
axes[1].imshow(Y_demo, cmap="gray")
axes[1].set_title("Grayscale")

# SLIC
axes[2].imshow(slic_view)
axes[2].set_title("SLIC Superpixels")

# SVR
axes[3].imshow(svr_rgb)
axes[3].set_title("SVR Prediction")

# SVR + MRF
axes[4].imshow(mrf_rgb)
axes[4].set_title("SVR + MRF")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("===== FINAL DEMO COMPLETE =====")
print("Image:", os.path.basename(demo_path))
print("SLIC regions:", len(segment_ids))
print("Features:", X_demo_24.shape)

In [ ]:
# ============================================================
# FINAL — UPLOAD IMAGE + COMPLETE DEMO
# ============================================================

from google.colab import files
import os

# Upload image
uploaded = files.upload()

# Get uploaded filename
demo_filename = next(iter(uploaded.keys()))
demo_path = demo_filename

print("Using uploaded image:", demo_filename)

# ============================================================
# 1. LOAD IMAGE
# ============================================================

image_bgr = cv2.imread(demo_path)

if image_bgr is None:
    raise ValueError("Could not read the uploaded image.")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_rgb = image_rgb.astype(np.float32) / 255.0

# Resize
new_width = 400
new_height = int(
    image_rgb.shape[0] * new_width / image_rgb.shape[1]
)

image_rgb = cv2.resize(
    image_rgb,
    (new_width, new_height),
    interpolation=cv2.INTER_AREA
)

# ============================================================
# 2. YUV
# ============================================================

image_yuv = cv2.cvtColor(
    image_rgb,
    cv2.COLOR_RGB2YUV
)

Y_demo = image_yuv[:, :, 0]

# ============================================================
# 3. SLIC
# ============================================================

segments_demo = slic(
    image_rgb,
    n_segments=100,
    compactness=10,
    start_label=0
)

segment_ids = np.unique(segments_demo)

slic_view = mark_boundaries(
    image_rgb,
    segments_demo,
    color=(1, 1, 0),
    mode="outer"
)

# ============================================================
# 4. EXTRACT 236 FEATURES
# ============================================================

X_demo_236 = []

for segment_id in segment_ids:

    features = extract_features_for_segment(
        Y_demo,
        segments_demo,
        segment_id
    )

    X_demo_236.append(features)

X_demo_236 = np.array(X_demo_236)

# ============================================================
# 5. CREATE FINAL 24 FEATURES
# ============================================================

X_demo_20 = X_demo_236[:, selected_features_20]

raw_demo_patches = X_demo_236[:, :100].reshape(-1, 10, 10)

top_left = np.mean(
    raw_demo_patches[:, :5, :5],
    axis=(1, 2)
)

top_right = np.mean(
    raw_demo_patches[:, :5, 5:],
    axis=(1, 2)
)

bottom_left = np.mean(
    raw_demo_patches[:, 5:, :5],
    axis=(1, 2)
)

bottom_right = np.mean(
    raw_demo_patches[:, 5:, 5:],
    axis=(1, 2)
)

local_demo_features = np.column_stack([
    top_left,
    top_right,
    bottom_left,
    bottom_right
])

X_demo_24 = np.hstack([
    X_demo_20,
    local_demo_features
])

# ============================================================
# 6. SCALE
# ============================================================

X_demo_scaled = v2_final_scaler.transform(
    X_demo_24
)

# ============================================================
# 7. SVR PREDICTION
# ============================================================

U_pred_demo = v2_final_svr_U.predict(
    X_demo_scaled
)

V_pred_demo = v2_final_svr_V.predict(
    X_demo_scaled
)

# ============================================================
# 8. SVR RECONSTRUCTION
# ============================================================

U_svr = np.zeros_like(Y_demo)
V_svr = np.zeros_like(Y_demo)

for i, segment_id in enumerate(segment_ids):

    mask = segments_demo == segment_id

    U_svr[mask] = U_pred_demo[i]
    V_svr[mask] = V_pred_demo[i]

svr_yuv = np.stack([
    Y_demo,
    U_svr,
    V_svr
], axis=2)

svr_rgb = cv2.cvtColor(
    svr_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

svr_rgb = np.clip(
    svr_rgb,
    0,
    1
)

# ============================================================
# 9. BUILD NEIGHBOR GRAPH
# ============================================================

neighbors = {
    int(s): set()
    for s in segment_ids
}

h, w = segments_demo.shape

for y in range(h):

    for x in range(w):

        current = int(
            segments_demo[y, x]
        )

        if x + 1 < w:

            right = int(
                segments_demo[y, x + 1]
            )

            if current != right:

                neighbors[current].add(right)
                neighbors[right].add(current)

        if y + 1 < h:

            bottom = int(
                segments_demo[y + 1, x]
            )

            if current != bottom:

                neighbors[current].add(bottom)
                neighbors[bottom].add(current)

# ============================================================
# 10. MRF / ICM
# ============================================================

U_mrf = U_pred_demo.copy()
V_mrf = V_pred_demo.copy()

MRF_GAMMA = 5
MRF_ITERATIONS = 5

segment_to_index = {
    int(segment_id): i
    for i, segment_id in enumerate(segment_ids)
}

for iteration in range(MRF_ITERATIONS):

    for i, segment_id in enumerate(segment_ids):

        neighbour_ids = neighbors[
            int(segment_id)
        ]

        if len(neighbour_ids) == 0:
            continue

        neighbour_indices = [
            segment_to_index[int(n)]
            for n in neighbour_ids
        ]

        mean_U = np.mean(
            U_mrf[neighbour_indices]
        )

        mean_V = np.mean(
            V_mrf[neighbour_indices]
        )

        U_mrf[i] = (
            U_mrf[i]
            + MRF_GAMMA * mean_U
        ) / (1 + MRF_GAMMA)

        V_mrf[i] = (
            V_mrf[i]
            + MRF_GAMMA * mean_V
        ) / (1 + MRF_GAMMA)

# ============================================================
# 11. MRF RECONSTRUCTION
# ============================================================

U_mrf_image = np.zeros_like(
    Y_demo,
    dtype=np.float32
)

V_mrf_image = np.zeros_like(
    Y_demo,
    dtype=np.float32
)

for i, segment_id in enumerate(segment_ids):

    mask = segments_demo == segment_id

    U_mrf_image[mask] = U_mrf[i]
    V_mrf_image[mask] = V_mrf[i]

mrf_yuv = np.stack([
    Y_demo,
    U_mrf_image,
    V_mrf_image
], axis=2)

mrf_rgb = cv2.cvtColor(
    mrf_yuv.astype(np.float32),
    cv2.COLOR_YUV2RGB
)

mrf_rgb = np.clip(
    mrf_rgb,
    0,
    1
)

# ============================================================
# 12. FINAL 5-PANEL DEMO
# ============================================================

fig, axes = plt.subplots(
    1,
    5,
    figsize=(22, 5)
)

axes[0].imshow(image_rgb)
axes[0].set_title("Original")

axes[1].imshow(
    Y_demo,
    cmap="gray"
)
axes[1].set_title("Grayscale")

axes[2].imshow(slic_view)
axes[2].set_title("SLIC Superpixels")

axes[3].imshow(svr_rgb)
axes[3].set_title("SVR Prediction")

axes[4].imshow(mrf_rgb)
axes[4].set_title("SVR + MRF")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# FINAL INFORMATION
# ============================================================

print("========================================")
print("       FINAL DEMO COMPLETE")
print("========================================")
print("Input image       :", demo_filename)
print("Image size        :", image_rgb.shape)
print("SLIC regions      :", len(segment_ids))
print("Original features :", X_demo_236.shape[1])
print("Final features    :", X_demo_24.shape[1])
print("========================================")